# 02 — Alpha Diversity (SGA vs AGA and FGR vs AGA)

**Manuscript:** Associations between gut microbiota diversity in early and late pregnancy and small for gestational age or growth restricted births: a nested case-control study

**Purpose:** Computes observed species richness, Shannon diversity and Pielou's evenness from MetaPhlAn4 relative abundances. Compares SGA vs AGA and FGR vs AGA at TP1 and TP2 (unadjusted t-tests and confounder-adjusted logistic regression). 

**Produces:** Figure 2 (unadjusted alpha diversity at TP1 & TP2), Tbale 2, Figure 3(adjusted alpha diversity at TP1 & TP2) and Table 3

**Programmer:** Anusha Antony

In [ ]:
library(dplyr)
library(tidyverse)
library(vegan)
library(ggplot2)
library(patchwork)
library(stringr)
library(patchwork)
library(cowplot)

In [ ]:
# Define base path 
base_path_1 <- "/proj/h-mel-phylo/h-mel-phylo/private/anusha/SGA_FGR/clean_code_version_260710/Git_hub_combined"
base_path_2 <- "/home/anusha/SGA-vs-AGA/data"
base_path_3 <- "/home/anusha/FGR-vs-AGA_260218/data"

In [ ]:
read_tab <- read.table("data")
head(read_tab)

## SGA vs AGA
Comparing the gut microbial diversity between mothers who delivered SGA neonates and those who delivered AGA neonates.
Sample Size: 
Case-SGA - 171
Control:342 

In [ ]:
SGA_meta_tp1 <- read.csv(file.path(base_path_2, "data"))
head(SGA_meta_tp1)
dim(SGA_meta_tp1)

In [ ]:
FGR_meta_tp1 <- read.csv(file.path(base_path_3, "data"))
head(FGR_meta_tp1)
dim(FGR_meta_tp1)


In [ ]:
length(intersect(FGR_meta_tp1$kit1.faecal_sample.barcode, SGA_meta_tp1$kit1.faecal_sample.barcode))

In [ ]:
SGA_tp1_taxa <- read.table(file.path(base_path_2, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)
head(SGA_tp1_taxa)
dim(SGA_tp1_taxa )

In [ ]:
#colSums(SGA_tp1_taxa)

In [ ]:
# Classified taxa rarely sum to 100% per sample, since low-abundance taxa are dropped during filtering. 
#This adds an "Other_taxa" row with the missing fraction so every sample sums to 100% for plotting and comparison.
add_other_taxa <- function(taxa_table, min_total = 100, row_name = "Other_taxa") {
  
  col_totals <- colSums(taxa_table)
  new_row <- ifelse(col_totals < min_total,
                    min_total - col_totals,
                    0)

  new_row <- setNames(new_row, names(taxa_table))
  
  # Convert to one-row data frame
  other_taxa_row <- as.data.frame(t(new_row))
  rownames(other_taxa_row) <- row_name
  
  # Add row to original table
  new_table <- rbind(taxa_table, other_taxa_row)
  
  # Return result
  return(as.data.frame(new_table))
}

In [ ]:
SGA_tp1_taxa_full <- add_other_taxa(SGA_tp1_taxa)
head(SGA_tp1_taxa_full)

In [ ]:
filt_SGA_meta_tp1 <- SGA_meta_tp1%>% select(Q1_Personnummer, Studienummer, kit1.faecal_sample.barcode,kit2.faecal_sample.barcode,age_checked, Age_groups, ATA_BMI_group_prior,Q1_X62_Alcohol_3_months_before_pregnancy,
         Primipara, Q1_X31_Diagnosed_PCOS,,Q1_X75_prior_bloodpressure_medication,Q1_X77_during_bloodpressure_medication,Q2_X15_during_bloodpressure_medication,Insufficient_weight_gain,preeclampsia_health_reg_ATA, Group)    
head(filt_SGA_meta_tp1)

In [ ]:
filt_SGA_meta_tp1$Age_above_35 <- with(filt_SGA_meta_tp1,
  ifelse(is.na(Age_groups), NA, ifelse(Age_groups %in% c(4,5), "1", "0")))
head(filt_SGA_meta_tp1)

In [ ]:
table(filt_SGA_meta_tp1$Group)

In [ ]:
filt_SGA_meta_tp1$Q1_X31_Diagnosed_PCOS <- with(filt_SGA_meta_tp1,
  ifelse(is.na(Q1_X31_Diagnosed_PCOS), NA_character_,
    ifelse(Q1_X31_Diagnosed_PCOS == "Ja", "1",
      ifelse(Q1_X31_Diagnosed_PCOS == "Nej", "0",
        ifelse(Q1_X31_Diagnosed_PCOS == "Vet inte", NA_character_, NA_character_)))))

In [ ]:
table(filt_SGA_meta_tp1$Q1_X31_Diagnosed_PCOS)
sum(is.na(filt_SGA_meta_tp1$Q1_X31_Diagnosed_PCOS))

In [ ]:
# ensure factors exist and set reference explicitly
filt_SGA_meta_tp1$Age_above_35 <- factor(filt_SGA_meta_tp1$Age_above_35, levels = c("0","1"))

filt_SGA_meta_tp1$ATA_BMI_group_prior <- factor(filt_SGA_meta_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
filt_SGA_meta_tp1$Primipara <- factor(filt_SGA_meta_tp1$Primipara, levels = c("0","1"))

filt_SGA_meta_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(filt_SGA_meta_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
filt_SGA_meta_tp1$Q1_X31_Diagnosed_PCOS <- factor(filt_SGA_meta_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

filt_SGA_meta_tp1$Insufficient_weight_gain <- factor(filt_SGA_meta_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

filt_SGA_meta_tp1$Q2_X15_during_bloodpressure_medication <- factor(filt_SGA_meta_tp1$Q2_X15_during_bloodpressure_medication,
                                                             levels = c("0","1"))
filt_SGA_meta_tp1$Q1_X77_during_bloodpressure_medication <- factor(filt_SGA_meta_tp1$Q1_X77_during_bloodpressure_medication,levels = c("0","1")) 
filt_SGA_meta_tp1$preeclampsia_health_reg_ATA <- factor(filt_SGA_meta_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
filt_SGA_meta_tp1$Group <- factor(filt_SGA_meta_tp1$Group ,
                                                             levels = c("Control","Case-SGA"))


In [ ]:
head(filt_SGA_meta_tp1)

In [ ]:
write.csv(
  filt_SGA_meta_tp1,
  file.path(base_path_1, "data")
)

In [ ]:
small_meta_tp2 <- read.csv(file.path(base_path_2,"data"),
                               header = TRUE)
head(small_meta_tp2 )
dim(small_meta_tp2)

In [ ]:
filt_SGA_meta_tp2 <- filt_SGA_meta_tp1 %>% filter(kit2.faecal_sample.barcode %in% small_meta_tp2$kit2.faecal_sample.barcode)
dim(filt_SGA_meta_tp2)
head(filt_SGA_meta_tp2 )

In [ ]:
write.csv(
  filt_SGA_meta_tp2,
  file.path(base_path_1, "data")
)

In [ ]:
SGA_tp2_taxa <- read.table(file.path(base_path_2, "tdata") ,sep= "\t",
  header      = TRUE,
  check.names = FALSE)
head(SGA_tp2_taxa )

In [ ]:
#colSums(SGA_tp2_taxa)

In [ ]:
SGA_tp2_taxa_full <- add_other_taxa(SGA_tp2_taxa)
head(SGA_tp2_taxa_full)

In [ ]:
#colSums(SGA_tp2_taxa_full)

In [ ]:
write_tsv(
  rownames_to_column(SGA_tp2_taxa_full, var = "taxa"),
  file.path(base_path_1, "data")
)

### 1. Calculate , Shannon,richness, evenness (TP1, TP2)

#### TP1

In [ ]:
SGA_case_df_1 <- filt_SGA_meta_tp1  %>%
  filter(Group == "Case-SGA")

SGA_case_kit1 <- SGA_case_df_1  %>%
  pull("kit1.faecal_sample.barcode")

In [ ]:
SGA_control_df_1 <- filt_SGA_meta_tp1 %>%
  filter(Group == "Control")

   SGA_control_kit1 <- SGA_control_df_1 %>%
  pull("kit1.faecal_sample.barcode")
 

In [ ]:
SGA_Case_taxa_1 <- SGA_tp1_taxa_full[,colnames(SGA_tp1_taxa_full) %in% SGA_case_kit1]
dim(SGA_Case_taxa_1)

SGA_Control_taxa_1 <- SGA_tp1_taxa_full[,colnames(SGA_tp1_taxa_full) %in% SGA_control_kit1]
dim(SGA_Control_taxa_1)


In [ ]:
#Shannon
SGA_case_shan_1 <- apply(SGA_Case_taxa_1, 2, diversity, index = "shannon")

SGA_control_shan_1 <- apply(SGA_Control_taxa_1, 2, diversity, index = "shannon")


In [ ]:
#calculates species richness 
SGA_case_rich_1 <- apply(SGA_Case_taxa_1, 2, specnumber)
SGA_control_rich_1 <- apply(SGA_Control_taxa_1, 2, specnumber)

In [ ]:
#Evenness: Pielou
SGA_case_even_1 <- SGA_case_shan_1/log(SGA_case_rich_1)
SGA_control_even_1 <- SGA_control_shan_1/log(SGA_control_rich_1)

#### TP2

In [ ]:
SGA_case_df_2 <- filt_SGA_meta_tp2  %>%
  filter(Group == "Case-SGA")

  SGA_case_kit2 <- SGA_case_df_2  %>%
  pull("kit2.faecal_sample.barcode")


In [ ]:
SGA_control_df_2 <- filt_SGA_meta_tp2  %>%
  filter(Group == "Control")

   SGA_control_kit2 <- SGA_control_df_2 %>%
  pull("kit2.faecal_sample.barcode")


In [ ]:
SGA_Case_taxa_2 <- SGA_tp2_taxa_full[,colnames(SGA_tp2_taxa_full) %in% SGA_case_kit2 ]
dim(SGA_Case_taxa_2)

SGA_Control_taxa_2 <- SGA_tp2_taxa_full[,colnames(SGA_tp2_taxa_full) %in% SGA_control_kit2]
dim(SGA_Control_taxa_2)

In [ ]:
#Shannon
SGA_case_shan_2 <- apply(SGA_Case_taxa_2, 2, diversity, index = "shannon")

SGA_control_shan_2 <- apply(SGA_Control_taxa_2, 2, diversity, index = "shannon")


In [ ]:
#calculates species richness 
SGA_case_rich_2 <- apply(SGA_Case_taxa_2, 2, specnumber)
SGA_control_rich_2 <- apply(SGA_Control_taxa_2, 2, specnumber)

In [ ]:
#Evenness: Pielou
SGA_case_even_2 <- SGA_case_shan_2/log(SGA_case_rich_2)
SGA_control_even_2 <- SGA_control_shan_2/log(SGA_control_rich_2)

#### Arrange the data frame for further analysis. 

In [ ]:
identical(
  names(SGA_case_shan_1),
  names(SGA_case_rich_1)
) &&
identical(
  names(SGA_case_shan_1),
  names(SGA_case_even_1 )
)

In [ ]:
identical(
  names(SGA_control_shan_2),
  names(SGA_control_rich_2)
) &&
identical(
  names(SGA_control_shan_2),
  names(SGA_control_even_2 )
)

In [ ]:
SGA_alpha_tp1 <- data.frame(
  kit1.faecal_sample.barcode = c(names(SGA_case_shan_1), names(SGA_control_shan_1)),
  Group = rep(
    c("Case-SGA", "Control"),
    times = c(length(SGA_case_shan_1), length(SGA_control_shan_1))),
  Shannon = c(SGA_case_shan_1, SGA_control_shan_1),
  Richness = c(SGA_case_rich_1, SGA_control_rich_1),
  Evenness = c(SGA_case_even_1 , SGA_control_even_1 )
)

rownames(SGA_alpha_tp1) <- NULL
head(SGA_alpha_tp1 )

In [ ]:
# Calculate Z-scores
SGA_alpha_tp1$Shannon_Z  <- as.numeric(scale(SGA_alpha_tp1$Shannon))
SGA_alpha_tp1$Richness_Z <- as.numeric(scale(SGA_alpha_tp1$Richness))
SGA_alpha_tp1$Evenness_Z <- as.numeric(scale(SGA_alpha_tp1$Evenness))
head(SGA_alpha_tp1)
dim(SGA_alpha_tp1)

In [ ]:
dim(SGA_alpha_tp1)

In [ ]:
filt_SGA_meta_tp1$kit1.faecal_sample.barcode <- as.character(filt_SGA_meta_tp1$kit1.faecal_sample.barcode)

In [ ]:
final_SGA_alpha_tp1 <- filt_SGA_meta_tp1 %>%
  select(-Group) %>%
  left_join(
    SGA_alpha_tp1,
    by = "kit1.faecal_sample.barcode"
  )
head(final_SGA_alpha_tp1)
dim(final_SGA_alpha_tp1)

In [ ]:
final_SGA_alpha_tp1$Group <- as.factor(final_SGA_alpha_tp1$Group)
final_SGA_alpha_tp1$Group <- relevel(final_SGA_alpha_tp1$Group, ref = "Control")

In [ ]:
SGA_alpha_tp2 <- data.frame(
  kit2.faecal_sample.barcode = c(names(SGA_case_shan_2), names(SGA_control_shan_2)),
  Group = rep(
    c("Case-SGA", "Control"),
    times = c(length(SGA_case_shan_2), length(SGA_control_shan_2))),
  Shannon = c(SGA_case_shan_2, SGA_control_shan_2),
  Richness = c(SGA_case_rich_2, SGA_control_rich_2),
  Evenness = c(SGA_case_even_2 , SGA_control_even_2 )
)

rownames(SGA_alpha_tp2) <- NULL
head(SGA_alpha_tp2 )

In [ ]:
# Calculate Z-scores
SGA_alpha_tp2$Shannon_Z  <- as.numeric(scale(SGA_alpha_tp2$Shannon))
SGA_alpha_tp2$Richness_Z <- as.numeric(scale(SGA_alpha_tp2$Richness))
SGA_alpha_tp2$Evenness_Z <- as.numeric(scale(SGA_alpha_tp2$Evenness))
head(SGA_alpha_tp2)
dim(SGA_alpha_tp2)

In [ ]:
filt_SGA_meta_tp2$kit2.faecal_sample.barcode <- as.character(filt_SGA_meta_tp2$kit2.faecal_sample.barcode)

In [ ]:
final_SGA_alpha_tp2 <- filt_SGA_meta_tp2 %>%
  select(-Group) %>%
  left_join(
    SGA_alpha_tp2,
    by = "kit2.faecal_sample.barcode"
  )
head(final_SGA_alpha_tp2)
dim(final_SGA_alpha_tp2)

In [ ]:
final_SGA_alpha_tp2$Group <- as.factor(final_SGA_alpha_tp2$Group)
final_SGA_alpha_tp2$Group <- relevel(final_SGA_alpha_tp2$Group, ref = "Control")

#### Unadjusted analysis

##### TP1

In [ ]:
#Shannon diversity: Raw values
# Shannon diversity - Case
hist(
  final_SGA_alpha_tp1$Shannon[final_SGA_alpha_tp1$Group == "Case-SGA"],
  main = "Histogram of Case-SGA Group",
  xlab = "Shannon diversity",
  col = "#4A9ED9",
  breaks = 20
)

# Shannon diversity - Control
hist(
  final_SGA_alpha_tp1$Shannon[final_SGA_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Shannon diversity",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_shannon_tp1 <- var.test(Shannon ~ Group, data = final_SGA_alpha_tp1)
SGA_var_shannon_tp1 
#Equal variance: If p-value >= 0.05, assume equal variances and use the pooled t-test(Student t test)


In [ ]:
t.test(Shannon ~ Group,data = final_SGA_alpha_tp1,var.equal = TRUE)

In [ ]:
p_value_SGA_shan_tp1 <- round(t.test(Shannon ~ Group,data = final_SGA_alpha_tp1,var.equal = TRUE)$p.value, 3)
p_value_SGA_shan_tp1

In [ ]:
# Shannon diversity: Z-transformed 
Z_crude_shan_SGA_tp1 <- glm(Group ~ Shannon_Z, data = final_SGA_alpha_tp1,family = binomial)
summary(Z_crude_shan_SGA_tp1)

In [ ]:
#Richness: Raw values

In [ ]:
# Richness - Case
hist(
  final_SGA_alpha_tp1$Richness[final_SGA_alpha_tp1$Group == "Case-SGA"],
  main = "Histogram of Case Group",
  xlab = "Richness",
  col = "#4A9ED9",
  breaks = 20
)

# Richness - Control
hist(
  final_SGA_alpha_tp1$Richness[final_SGA_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Richness",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_rich_tp1 <- var.test(Richness ~ Group, data = final_SGA_alpha_tp1)
SGA_var_rich_tp1 
#Equal variance: If p-value >= 0.05, assume equal variances and use the pooled t-test(Student t test)

In [ ]:
t.test(Richness  ~ Group,data = final_SGA_alpha_tp1,var.equal = TRUE)

In [ ]:
p_value_SGA_rich_tp1 <- round(t.test(Richness  ~ Group,data = final_SGA_alpha_tp1,var.equal = TRUE)$p.value, 3)
p_value_SGA_rich_tp1

In [ ]:
# species richness: Z-transformed 
Z_crude_rich_SGA_tp1 <- glm(Group ~ Richness_Z, data = final_SGA_alpha_tp1,family = binomial)
summary(Z_crude_rich_SGA_tp1)

In [ ]:
#Evenness
# Evenness - Case
hist(
  final_SGA_alpha_tp1$Evenness[final_SGA_alpha_tp1$Group == "Case-SGA"],
  main = "Histogram of Case Group",
  xlab = "Evenness",
  col = "#4A9ED9",
  breaks = 20
)

# Evenness - Control
hist(
  final_SGA_alpha_tp1$Evenness[final_SGA_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Evenness",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_even_tp1 <- var.test(Evenness ~ Group, data = final_SGA_alpha_tp1) 
SGA_var_even_tp1

In [ ]:
t.test(Evenness ~ Group, data = final_SGA_alpha_tp1, var.equal = TRUE)

In [ ]:
p_value_SGA_even_tp1 <- round(t.test(Evenness ~ Group, data = final_SGA_alpha_tp1, var.equal = TRUE)$p.value, 3)
p_value_SGA_even_tp1

In [ ]:
Z_crude_even_SGA_tp1 <- glm(Group ~ Evenness_Z, data = final_SGA_alpha_tp1,family = binomial)
summary(Z_crude_even_SGA_tp1 )

##### TP2

In [ ]:
#Shannon diversity: Raw values
# Shannon diversity - Case
hist(
  final_SGA_alpha_tp2$Shannon[final_SGA_alpha_tp2$Group == "Case-SGA"],
  main = "Histogram of Case-SGA Group",
  xlab = "Shannon diversity",
  col = "#4A9ED9",
  breaks = 20
)

# Shannon diversity - Control
hist(
  final_SGA_alpha_tp2$Shannon[final_SGA_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Shannon diversity",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_shannon_tp2 <- var.test(Shannon ~ Group, data = final_SGA_alpha_tp2)
SGA_var_shannon_tp2 

In [ ]:
t.test(Shannon ~ Group,data = final_SGA_alpha_tp2,var.equal = TRUE)

In [ ]:
p_value_SGA_shan_tp2 <- round(t.test(Shannon ~ Group,data = final_SGA_alpha_tp2,var.equal = TRUE)$p.value, 3)
p_value_SGA_shan_tp2

In [ ]:
# Shannon diversity: Z-transformed 
Z_crude_shan_SGA_tp2 <- glm(Group ~ Shannon_Z, data = final_SGA_alpha_tp2,family = binomial)
summary(Z_crude_shan_SGA_tp2)

In [ ]:
# Richness - Case
hist(
  final_SGA_alpha_tp2$Richness[final_SGA_alpha_tp2$Group == "Case-SGA"],
  main = "Histogram of Case Group",
  xlab = "Richness",
  col = "#4A9ED9",
  breaks = 20
)

# Richness - Control
hist(
  final_SGA_alpha_tp2$Richness[final_SGA_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Richness",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_rich_tp2 <- var.test(Richness ~ Group, data = final_SGA_alpha_tp2)
SGA_var_rich_tp2 

In [ ]:
t.test(Richness  ~ Group,data = final_SGA_alpha_tp2,var.equal = TRUE)

In [ ]:
p_value_SGA_rich_tp2 <- round(t.test(Richness  ~ Group,data = final_SGA_alpha_tp2,var.equal = TRUE)$p.value, 3)
p_value_SGA_rich_tp2

In [ ]:
# species richness: Z-transformed 
Z_crude_rich_SGA_tp2 <- glm(Group ~ Richness_Z, data = final_SGA_alpha_tp2,family = binomial)
summary(Z_crude_rich_SGA_tp2)

In [ ]:
#Evenness
# Evenness - Case
hist(
  final_SGA_alpha_tp2$Evenness[final_SGA_alpha_tp2$Group == "Case-SGA"],
  main = "Histogram of Case Group",
  xlab = "Evenness",
  col = "#4A9ED9",
  breaks = 20
)

# Evenness - Control
hist(
  final_SGA_alpha_tp2$Evenness[final_SGA_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Evenness",
  col = "#708090",
  breaks = 20
)

In [ ]:
SGA_var_even_tp2 <- var.test(Evenness ~ Group, data = final_SGA_alpha_tp2) 
SGA_var_even_tp2

In [ ]:
t.test(Evenness ~ Group, data = final_SGA_alpha_tp2, var.equal = TRUE)

In [ ]:
p_value_SGA_even_tp2 <- round(t.test(Evenness ~ Group, data = final_SGA_alpha_tp2, var.equal = TRUE)$p.value, 3)
p_value_SGA_even_tp2

In [ ]:
Z_crude_even_SGA_tp2 <- glm(Group ~ Evenness_Z, data = final_SGA_alpha_tp2,family = binomial)
summary(Z_crude_even_SGA_tp2)

#### Adjusted analysis
Adjusting for confounders: TP1: Advanced maternal age (≥35 years), Pre-pregnancy BMI (categorized as ≤18, 19–24, 25–30, or >30 kg/m²),
Alcohol consumption (in the 3 months prior to pregnancy), Nulliparity (yes/no), Polycystic ovary syndrome (PCOS) (yes/no), Insufficient gestational weight gain (yes/no), Blood pressure medication use during pregnancy (yes/no)

TP2: TP1 confounders and Preeclampsia: Adjusted for at TP2 as a potential mediator, but excluded at TP1 because preeclampsia is diagnosed after 20 weeks of gestation


##### TP1

In [ ]:
model_SGA_tp1_shan <- glm(Group ~ Shannon_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_SGA_alpha_tp1,
                   family = binomial)
summary(model_SGA_tp1_shan)

In [ ]:
model_SGA_tp1_rich <- glm(Group ~ Richness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_SGA_alpha_tp1,
                   family = binomial)
summary(model_SGA_tp1_rich )

In [ ]:
model_SGA_tp1_even <- glm(Group ~ Evenness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_SGA_alpha_tp1,
                   family = binomial)
summary(model_SGA_tp1_even)

##### TP2

In [ ]:
model_SGA_tp2_shan <- glm(Group ~ Shannon_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication +  preeclampsia_health_reg_ATA,
                   data = final_SGA_alpha_tp2,
                   family = binomial)
summary(model_SGA_tp2_shan )

In [ ]:
model_SGA_tp2_rich <- glm(Group ~ Richness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication + preeclampsia_health_reg_ATA,
                   data = final_SGA_alpha_tp2,
                   family = binomial)
summary(model_SGA_tp2_rich)

In [ ]:
model_SGA_tp2_even <- glm(Group ~ Evenness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication + preeclampsia_health_reg_ATA,
                   data = final_SGA_alpha_tp2,
                   family = binomial)
summary(model_SGA_tp2_even )

## FGR vs AGA
Comparing the gut microbial diversity between mothers who delivered FGR neonates and those who delivered AGA neonates.
Sample Size: 
Case-FGR - 171
Control:342 

In [ ]:
FGR_meta_tp1 <- read.csv(file.path(base_path_3, "data"))
head(FGR_meta_tp1)
dim(FGR_meta_tp1)


In [ ]:
FGR_meta_tp2 <- read.csv(file.path(base_path_3, "data"))
head(FGR_meta_tp2)
dim(FGR_meta_tp2)

In [ ]:
FGR_tp1_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)
head(FGR_tp1_taxa)
dim(FGR_tp1_taxa )

In [ ]:
FGR_tp2_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)
head(FGR_tp2_taxa)
dim(FGR_tp2_taxa )

In [ ]:
FGR_tp1_taxa_full <- add_other_taxa(FGR_tp1_taxa)
head(FGR_tp1_taxa_full)

In [ ]:
#colSums(FGR_tp1_taxa_full)

In [ ]:
FGR_tp2_taxa_full <- add_other_taxa(FGR_tp2_taxa)
head(FGR_tp2_taxa_full)

In [ ]:
#colSums(FGR_tp2_taxa_full)

In [ ]:
filt_FGR_meta_tp1 <- FGR_meta_tp1%>% select(Q1_Personnummer, Studienummer, kit1.faecal_sample.barcode,kit2.faecal_sample.barcode,age_checked, Age_groups, ATA_BMI_group_prior,Q1_X62_Alcohol_3_months_before_pregnancy,
         Primipara, Q1_X31_Diagnosed_PCOS,,Q1_X75_prior_bloodpressure_medication,Q1_X77_during_bloodpressure_medication,Q2_X15_during_bloodpressure_medication,Insufficient_weight_gain,preeclampsia_health_reg_ATA, Group)    
head(filt_FGR_meta_tp1)

In [ ]:
filt_FGR_meta_tp1$Age_above_35 <- with(filt_FGR_meta_tp1,
  ifelse(is.na(Age_groups), NA, ifelse(Age_groups %in% c(4,5), "1", "0")))
head(filt_FGR_meta_tp1)


In [ ]:
filt_FGR_meta_tp1$Q1_X31_Diagnosed_PCOS <- with(filt_FGR_meta_tp1,
  ifelse(is.na(Q1_X31_Diagnosed_PCOS), NA_character_,
    ifelse(Q1_X31_Diagnosed_PCOS == "Ja", "1",
      ifelse(Q1_X31_Diagnosed_PCOS == "Nej", "0",
        ifelse(Q1_X31_Diagnosed_PCOS == "Vet inte", NA_character_, NA_character_)))))

In [ ]:
# ensure factors exist and set reference explicitly
filt_FGR_meta_tp1$Age_above_35 <- factor(filt_FGR_meta_tp1$Age_above_35, levels = c("0","1"))

filt_FGR_meta_tp1$ATA_BMI_group_prior <- factor(filt_FGR_meta_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
filt_FGR_meta_tp1$Primipara <- factor(filt_FGR_meta_tp1$Primipara, levels = c("0","1"))

filt_FGR_meta_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(filt_FGR_meta_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
filt_FGR_meta_tp1$Q1_X31_Diagnosed_PCOS <- factor(filt_FGR_meta_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

filt_FGR_meta_tp1$Insufficient_weight_gain <- factor(filt_FGR_meta_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

filt_FGR_meta_tp1$Q2_X15_during_bloodpressure_medication <- factor(filt_FGR_meta_tp1$Q2_X15_during_bloodpressure_medication,
                                                             levels = c("0","1"))
filt_FGR_meta_tp1$Q1_X77_during_bloodpressure_medication <- factor(filt_FGR_meta_tp1$Q1_X77_during_bloodpressure_medication,
                                                             levels = c("0","1"))
filt_FGR_meta_tp1$preeclampsia_health_reg_ATA <- factor(filt_FGR_meta_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
filt_FGR_meta_tp1$Group <- factor(filt_FGR_meta_tp1$Group ,
                                                             levels = c("Control","Case-FGR"))


In [ ]:
write.csv(
  filt_FGR_meta_tp1,
  file.path(base_path_1, "data")
)

In [ ]:
filt_FGR_meta_tp2 <- filt_FGR_meta_tp1 %>% filter(kit2.faecal_sample.barcode %in% FGR_meta_tp2 $kit2.faecal_sample.barcode)
dim(filt_FGR_meta_tp2)
head(filt_FGR_meta_tp2 )

In [ ]:
filt_FGR_meta_tp1$kit1.faecal_sample.barcode <- as.character(filt_FGR_meta_tp1$kit1.faecal_sample.barcode)
filt_FGR_meta_tp2$kit2.faecal_sample.barcode <- as.character(filt_FGR_meta_tp2$kit2.faecal_sample.barcode)

In [ ]:
write.csv(
  filt_FGR_meta_tp2,
  file.path(base_path_1, "data")
)

### 1. Calculate , Shannon,richness, evenness (TP1, TP2)

#### TP1

In [ ]:
FGR_case_df_1 <- filt_FGR_meta_tp1  %>%
  filter(Group == "Case-FGR")

FGR_case_kit1 <- FGR_case_df_1  %>%
  pull("kit1.faecal_sample.barcode")



In [ ]:
FGR_control_df_1 <- filt_FGR_meta_tp1 %>%
  filter(Group == "Control")

   FGR_control_kit1 <- FGR_control_df_1 %>%
  pull("kit1.faecal_sample.barcode")
 

In [ ]:
FGR_Case_taxa_1 <- FGR_tp1_taxa_full[,colnames(FGR_tp1_taxa_full) %in% FGR_case_kit1]
dim(FGR_Case_taxa_1)

FGR_Control_taxa_1 <- FGR_tp1_taxa_full[,colnames(FGR_tp1_taxa_full) %in% FGR_control_kit1]
dim(FGR_Control_taxa_1)

In [ ]:
#Shannon
FGR_case_shan_1 <- apply(FGR_Case_taxa_1, 2, diversity, index = "shannon")

FGR_control_shan_1 <- apply(FGR_Control_taxa_1, 2, diversity, index = "shannon")


In [ ]:
# Calculates species richness 
FGR_case_rich_1 <- apply(FGR_Case_taxa_1, 2, specnumber)
FGR_control_rich_1 <- apply(FGR_Control_taxa_1, 2, specnumber)

In [ ]:
#Evenness: Pielou
FGR_case_even_1 <- FGR_case_shan_1/log(FGR_case_rich_1)
FGR_control_even_1 <- FGR_control_shan_1/log(FGR_control_rich_1)

#### TP2

In [ ]:
FGR_case_df_2 <- filt_FGR_meta_tp2  %>%
  filter(Group == "Case-FGR")

FGR_case_kit2 <- FGR_case_df_2  %>%
  pull("kit2.faecal_sample.barcode")

In [ ]:
FGR_control_df_2 <- filt_FGR_meta_tp2  %>%
  filter(Group == "Control")

   FGR_control_kit2 <- FGR_control_df_2 %>%
  pull("kit2.faecal_sample.barcode")

In [ ]:
FGR_Case_taxa_2 <- FGR_tp2_taxa_full[,colnames(FGR_tp2_taxa_full) %in% FGR_case_kit2 ]
dim(FGR_Case_taxa_2)

FGR_Control_taxa_2 <- FGR_tp2_taxa_full[,colnames(FGR_tp2_taxa_full) %in% FGR_control_kit2]
dim(FGR_Control_taxa_2)


In [ ]:
#Shannon
FGR_case_shan_2 <- apply(FGR_Case_taxa_2, 2, diversity, index = "shannon")

FGR_control_shan_2 <- apply(FGR_Control_taxa_2, 2, diversity, index = "shannon")

In [ ]:
# Calculates species richness 
FGR_case_rich_2 <- apply(FGR_Case_taxa_2, 2, specnumber)
FGR_control_rich_2 <- apply(FGR_Control_taxa_2, 2, specnumber)


In [ ]:
#Evenness: Pielou
FGR_case_even_2 <- FGR_case_shan_2/log(FGR_case_rich_2)
FGR_control_even_2 <- FGR_control_shan_2/log(FGR_control_rich_2)



#### Arrange the data frame for further analysis. 


In [ ]:
identical(
  names(FGR_case_shan_1),
  names(FGR_case_rich_1)
) &&
identical(
  names(FGR_case_shan_1),
  names(FGR_case_even_1 )
)

In [ ]:
identical(
  names(FGR_control_shan_2),
  names(FGR_control_rich_2)
) &&
identical(
  names(FGR_control_shan_2),
  names(FGR_control_even_2 )
)

In [ ]:
FGR_alpha_tp1 <- data.frame(
  kit1.faecal_sample.barcode = c(names(FGR_case_shan_1), names(FGR_control_shan_1)),
  Group = rep(
    c("Case-FGR", "Control"),
    times = c(length(FGR_case_shan_1), length(FGR_control_shan_1))),
  Shannon = c(FGR_case_shan_1, FGR_control_shan_1),
  Richness = c(FGR_case_rich_1, FGR_control_rich_1),
  Evenness = c(FGR_case_even_1 , FGR_control_even_1 )
)


In [ ]:
rownames(FGR_alpha_tp1) <- NULL
head(FGR_alpha_tp1 )

In [ ]:
# Calculate Z-scores
FGR_alpha_tp1$Shannon_Z  <- as.numeric(scale(FGR_alpha_tp1$Shannon))
FGR_alpha_tp1$Richness_Z <- as.numeric(scale(FGR_alpha_tp1$Richness))
FGR_alpha_tp1$Evenness_Z <- as.numeric(scale(FGR_alpha_tp1$Evenness))
head(FGR_alpha_tp1)
dim(FGR_alpha_tp1)

In [ ]:
final_FGR_alpha_tp1 <- filt_FGR_meta_tp1 %>%
  select(-Group) %>%
  left_join(
    FGR_alpha_tp1,
    by = "kit1.faecal_sample.barcode"
  )
head(final_FGR_alpha_tp1)
dim(final_FGR_alpha_tp1)

In [ ]:
final_FGR_alpha_tp1$Group <- as.factor(final_FGR_alpha_tp1$Group)
final_FGR_alpha_tp1$Group <- relevel(final_FGR_alpha_tp1$Group, ref = "Control")

In [ ]:
FGR_alpha_tp2 <- data.frame(
  kit2.faecal_sample.barcode = c(names(FGR_case_shan_2), names(FGR_control_shan_2)),
  Group = rep(
    c("Case-FGR", "Control"),
    times = c(length(FGR_case_shan_2), length(FGR_control_shan_2))),
  Shannon = c(FGR_case_shan_2, FGR_control_shan_2),
  Richness = c(FGR_case_rich_2, FGR_control_rich_2),
  Evenness = c(FGR_case_even_2 , FGR_control_even_2 )
)


In [ ]:
rownames(FGR_alpha_tp2) <- NULL
head(FGR_alpha_tp2 )

In [ ]:
# Calculate Z-scores
FGR_alpha_tp2$Shannon_Z  <- as.numeric(scale(FGR_alpha_tp2$Shannon))
FGR_alpha_tp2$Richness_Z <- as.numeric(scale(FGR_alpha_tp2$Richness))
FGR_alpha_tp2$Evenness_Z <- as.numeric(scale(FGR_alpha_tp2$Evenness))
head(FGR_alpha_tp2)
dim(FGR_alpha_tp2)

In [ ]:

final_FGR_alpha_tp2 <- filt_FGR_meta_tp2 %>%
  select(-Group) %>%
  left_join(
    FGR_alpha_tp2,
    by = "kit2.faecal_sample.barcode"
  )
head(final_FGR_alpha_tp2)
dim(final_FGR_alpha_tp2)

In [ ]:
final_FGR_alpha_tp2$Group <- as.factor(final_FGR_alpha_tp2$Group)
final_FGR_alpha_tp2$Group <- relevel(final_FGR_alpha_tp2$Group, ref = "Control")

#### Unadjusted analysis

##### TP1

In [ ]:
#Shannon diversity: Raw values
# Shannon diversity - Case
hist(
  final_FGR_alpha_tp1$Shannon[final_FGR_alpha_tp1$Group == "Case-FGR"],
  main = "Histogram of Case-FGR Group",
  xlab = "Shannon diversity",
  col = "#C5808C",
  breaks = 20
)

# Shannon diversity - Control
hist(
  final_FGR_alpha_tp1$Shannon[final_FGR_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Shannon diversity",
  col = "#708090",
  breaks = 20
)

In [ ]:
#VARIANCE TEST
FGR_var_shannon_tp1 <- var.test(Shannon ~ Group, data = final_FGR_alpha_tp1)
FGR_var_shannon_tp1 

In [ ]:
t.test(Shannon ~ Group,data = final_FGR_alpha_tp1,var.equal = TRUE)

In [ ]:
p_value_FGR_shan_tp1 <- round(t.test(Shannon ~ Group,data = final_FGR_alpha_tp1,var.equal = TRUE)$p.value, 3)
p_value_FGR_shan_tp1


In [ ]:
# Shannon diversity: Z-transformed 
Z_crude_shan_FGR_tp1 <- glm(Group ~ Shannon_Z, data = final_FGR_alpha_tp1,family = binomial)
summary(Z_crude_shan_FGR_tp1)

In [ ]:
# Richness - Case
hist(
  final_FGR_alpha_tp1$Richness[final_FGR_alpha_tp1$Group == "Case-FGR"],
  main = "Histogram of Case Group",
  xlab = "Richness",
  col = "#C5808C",
  breaks = 20
)

# Richness - Control
hist(
  final_FGR_alpha_tp1$Richness[final_FGR_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Richness",
  col = "#708090",
  breaks = 20
)

In [ ]:
FGR_var_rich_tp1 <- var.test(Richness ~ Group, data = final_FGR_alpha_tp1)
FGR_var_rich_tp1 

In [ ]:
t.test(Richness  ~ Group,data = final_FGR_alpha_tp1,var.equal = TRUE)

In [ ]:
p_value_FGR_rich_tp1 <- round(t.test(Richness  ~ Group,data = final_FGR_alpha_tp1,var.equal = TRUE)$p.value, 3)
p_value_FGR_rich_tp1

In [ ]:
# species richness: Z-transformed 
Z_crude_rich_FGR_tp1 <- glm(Group ~ Richness_Z, data = final_FGR_alpha_tp1,family = binomial)
summary(Z_crude_rich_FGR_tp1)

In [ ]:
#Evenness
# Evenness - Case
hist(
  final_FGR_alpha_tp1$Evenness[final_FGR_alpha_tp1$Group == "Case-FGR"],
  main = "Histogram of Case Group",
  xlab = "Evenness",
  col = "#C5808C",
  breaks = 20
)

# Evenness - Control
hist(
  final_FGR_alpha_tp1$Evenness[final_FGR_alpha_tp1$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Evenness",
  col = "#708090",
  breaks = 20
)

In [ ]:
FGR_var_even_tp1 <- var.test(Evenness ~ Group, data = final_FGR_alpha_tp1) 
FGR_var_even_tp1

In [ ]:
p_value_FGR_even_tp1 <- round(t.test(Evenness ~ Group, data = final_FGR_alpha_tp1, var.equal = TRUE)$p.value, 3)
p_value_FGR_even_tp1

In [ ]:
Z_crude_even_FGR_tp1 <- glm(Group ~ Evenness_Z, data = final_FGR_alpha_tp1,family = binomial)
summary(Z_crude_even_FGR_tp1 )

##### TP2

In [ ]:
#Shannon diversity: Raw values
# Shannon diversity - Case
hist(
  final_FGR_alpha_tp2$Shannon[final_FGR_alpha_tp2$Group == "Case-FGR"],
  main = "Histogram of Case-FGR Group",
  xlab = "Shannon diversity",
  col = "#C5808C",
  breaks = 20
)

# Shannon diversity - Control
hist(
  final_FGR_alpha_tp2$Shannon[final_FGR_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Shannon diversity",
  col = "#708090",
  breaks = 20
)

In [ ]:
FGR_var_shannon_tp2 <- var.test(Shannon ~ Group, data = final_FGR_alpha_tp2)
FGR_var_shannon_tp2
#Equal variance: If p-value >= 0.05, assume equal variances and use the pooled t-test(Student t test)

In [ ]:
t.test(Shannon ~ Group,data = final_FGR_alpha_tp2,var.equal = FALSE)

In [ ]:
p_value_FGR_shan_tp2 <- round(t.test(Shannon ~ Group,data = final_FGR_alpha_tp2,var.equal = FALSE)$p.value, 3)
p_value_FGR_shan_tp2


In [ ]:
# Shannon diversity: Z-transformed 
Z_crude_shan_FGR_tp2 <- glm(Group ~ Shannon_Z, data = final_FGR_alpha_tp2,family = binomial)
summary(Z_crude_shan_FGR_tp2)

In [ ]:
# Richness - Case
hist(
  final_FGR_alpha_tp2$Richness[final_FGR_alpha_tp2$Group == "Case-FGR"],
  main = "Histogram of Case-FGR Group",
  xlab = "Richness",
  col = "#C5808C",
  breaks = 20
)

# Richness - Control
hist(
  final_FGR_alpha_tp2$Richness[final_FGR_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Richness",
  col = "#708090",
  breaks = 20
)


In [ ]:
FGR_var_rich_tp2 <- var.test(Richness ~ Group, data = final_FGR_alpha_tp2)
FGR_var_rich_tp2 

In [ ]:
t.test(Richness  ~ Group,data = final_FGR_alpha_tp2,var.equal = TRUE)

In [ ]:
p_value_FGR_rich_tp2 <- round(t.test(Richness  ~ Group,data = final_FGR_alpha_tp2,var.equal = TRUE)$p.value, 3)
p_value_FGR_rich_tp2


In [ ]:
# species richness: Z-transformed 
Z_crude_rich_FGR_tp2 <- glm(Group ~ Richness_Z, data = final_FGR_alpha_tp2,family = binomial)
summary(Z_crude_rich_FGR_tp2)


In [ ]:
#Evenness
# Evenness - Case
hist(
  final_FGR_alpha_tp2$Evenness[final_FGR_alpha_tp2$Group == "Case-FGR"],
  main = "Histogram of Case Group",
  xlab = "Evenness",
  col = "#C5808C",
  breaks = 20
)

# Evenness - Control
hist(
  final_FGR_alpha_tp2$Evenness[final_FGR_alpha_tp2$Group == "Control"],
  main = "Histogram of Control Group",
  xlab = "Evenness",
  col = "#708090",
  breaks = 20
)

In [ ]:
FGR_var_even_tp2 <- var.test(Evenness ~ Group, data = final_FGR_alpha_tp2) 
FGR_var_even_tp2


In [ ]:
t.test(Evenness ~ Group, data = final_FGR_alpha_tp2, var.equal = FALSE)

In [ ]:
p_value_FGR_even_tp2 <- round(t.test(Evenness ~ Group, data = final_FGR_alpha_tp2, var.equal = FALSE)$p.value, 3)
p_value_FGR_even_tp2

In [ ]:
Z_crude_even_FGR_tp2 <- glm(Group ~ Evenness_Z, data = final_FGR_alpha_tp2,family = binomial)
summary(Z_crude_even_FGR_tp2 )

#### Adjusted analysis

##### TP1

In [ ]:
model_FGR_tp1_shan <- glm(Group ~ Shannon_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_FGR_alpha_tp1,
                   family = binomial)
summary(model_FGR_tp1_shan)

In [ ]:
model_FGR_tp1_rich <- glm(Group ~ Richness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_FGR_alpha_tp1,
                   family = binomial)
summary(model_FGR_tp1_rich )

In [ ]:
model_FGR_tp1_even <- glm(Group ~ Evenness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q1_X77_during_bloodpressure_medication,
                   data = final_FGR_alpha_tp1,
                   family = binomial)
summary(model_FGR_tp1_even)

##### TP2

In [ ]:
model_FGR_tp2_shan <- glm(Group ~ Shannon_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication +  preeclampsia_health_reg_ATA,
                   data = final_FGR_alpha_tp2,
                   family = binomial)
summary(model_FGR_tp2_shan )

In [ ]:
model_FGR_tp2_rich <- glm(Group ~ Richness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication + preeclampsia_health_reg_ATA,
                   data = final_FGR_alpha_tp2,
                   family = binomial)
summary(model_FGR_tp2_rich)


In [ ]:
model_FGR_tp2_even <- glm(Group ~ Evenness_Z + Age_above_35 + ATA_BMI_group_prior + Primipara + Q1_X62_Alcohol_3_months_before_pregnancy
                        + Q1_X31_Diagnosed_PCOS + Insufficient_weight_gain + Q2_X15_during_bloodpressure_medication + preeclampsia_health_reg_ATA,
                   data = final_FGR_alpha_tp2,
                   family = binomial)
summary(model_FGR_tp2_even )

## Figures

### Data arrangements for the figures

In [ ]:
#Function
extract_stats <- function(model, exposure_name, risk, timepoint) {
  
  coef_summary <- summary(model)$coefficients
  
  est <- coef_summary[exposure_name, "Estimate"]
  se <- coef_summary[exposure_name, "Std. Error"]
  pval <- coef_summary[exposure_name, "Pr(>|z|)"]
  
  # OR and 95% CI
  OR <- exp(est)
  CI_lower <- exp(est - 1.96 * se)
  CI_upper <- exp(est + 1.96 * se)
  
  # Inverted OR and CI
  OR_inverted <- 1 / OR
  CI_lower_inverted <- 1 / CI_upper
  CI_upper_inverted <- 1 / CI_lower
  
  # P-value group
  p_group <- ifelse(
    pval > 0.05,
    "p > 0.05",
    "p <= 0.05"
  )
  
  return(data.frame(
    Risk = risk,
    Timepoint = timepoint,
    Alpha_diversity = exposure_name,
    OR = OR,
    CI_lower = CI_lower,
    CI_upper = CI_upper,
    OR_inverted = OR_inverted,
    CI_lower_inverted = CI_lower_inverted,
    CI_upper_inverted = CI_upper_inverted,
    p_value = pval,
    p_group = p_group
  ))
}

In [ ]:
#SGA
SGA_TP1_output <- bind_rows(
  extract_stats(model_SGA_tp1_shan, "Shannon_Z", "SGA", "TP1"),
  extract_stats(model_SGA_tp1_rich, "Richness_Z", "SGA", "TP1"),
  extract_stats(model_SGA_tp1_even, "Evenness_Z", "SGA", "TP1")
)

SGA_TP2_output <- bind_rows(
  extract_stats(model_SGA_tp2_shan, "Shannon_Z", "SGA", "TP2"),
  extract_stats(model_SGA_tp2_rich, "Richness_Z", "SGA", "TP2"),
  extract_stats(model_SGA_tp2_even, "Evenness_Z", "SGA", "TP2")
)

In [ ]:
#FGR
FGR_TP1_output <- bind_rows(
  extract_stats(model_FGR_tp1_shan, "Shannon_Z", "FGR", "TP1"),
  extract_stats(model_FGR_tp1_rich, "Richness_Z", "FGR", "TP1"),
  extract_stats(model_FGR_tp1_even, "Evenness_Z", "FGR", "TP1")
)

FGR_TP2_output <- bind_rows(
  extract_stats(model_FGR_tp2_shan, "Shannon_Z", "FGR", "TP2"),
  extract_stats(model_FGR_tp2_rich, "Richness_Z", "FGR", "TP2"),
  extract_stats(model_FGR_tp2_even, "Evenness_Z", "FGR", "TP2")
)

In [ ]:
adjust_alpha_output <- bind_rows(
  SGA_TP1_output,
  SGA_TP2_output,
  FGR_TP1_output,
  FGR_TP2_output
)
head(adjust_alpha_output)

In [ ]:


adjust_alpha_output <- adjust_alpha_output%>%
  mutate(Alpha_diversity = str_remove(Alpha_diversity, "_Z$"))

In [ ]:
head(adjust_alpha_output)

In [ ]:
adjust_alpha_output <- adjust_alpha_output%>%
  mutate(
    Risk = factor(Risk, levels = c("SGA", "FGR")),
    Timepoint = factor(Timepoint, levels = c("TP1", "TP2")),
    Alpha_diversity = factor(
      Alpha_diversity,
      levels = c("Shannon", "Richness", "Evenness")
    ),
    
    # Colour significant results by risk
    plot_colour = case_when(
      p_value <= 0.05 & Risk == "SGA" ~ "SGA p <= 0.05",
      p_value <= 0.05 & Risk == "FGR" ~ "FGR p <= 0.05",
      TRUE ~ "p > 0.05"
    )
  )

In [ ]:
#calculate a common y-axis range from all the CIs
y_min <- floor(
  min(adjust_alpha_output$CI_lower_inverted, na.rm = TRUE) * 4
) / 4

y_max <- ceiling(
  max(adjust_alpha_output$CI_upper_inverted, na.rm = TRUE) * 4
) / 4

In [ ]:
tags <- adjust_alpha_output |>
  dplyr::distinct(Risk, Timepoint) |>
  dplyr::arrange(Risk, Timepoint) |>
  dplyr::mutate(label = paste0(LETTERS[dplyr::row_number()], "."))

In [ ]:
head(final_SGA_alpha_tp1)

In [ ]:
final_SGA_alpha_tp1_fig <-  final_SGA_alpha_tp1 %>% select(kit1.faecal_sample.barcode, Shannon, Richness, Evenness, Group) %>%
                            mutate(Timepoint = "TP1") %>% rename(Sample_ID = kit1.faecal_sample.barcode)
head(final_SGA_alpha_tp1_fig)

In [ ]:
final_SGA_alpha_tp1_fig$Group <- relevel(final_SGA_alpha_tp1_fig$Group, ref = "Control")

In [ ]:
final_SGA_alpha_tp2_fig <-  final_SGA_alpha_tp2 %>% select(kit2.faecal_sample.barcode, Shannon, Richness, Evenness, Group) %>%
                            mutate(Timepoint = "TP2") %>% rename(Sample_ID = kit2.faecal_sample.barcode)
head(final_SGA_alpha_tp2_fig)

In [ ]:
final_SGA_alpha_tp2_fig$Group <- relevel(final_SGA_alpha_tp2_fig$Group, ref = "Control")

In [ ]:
final_FGR_alpha_tp1_fig <-  final_FGR_alpha_tp1 %>% select(kit1.faecal_sample.barcode, Shannon, Richness, Evenness, Group) %>%
                            mutate(Timepoint = "TP1") %>% rename(Sample_ID = kit1.faecal_sample.barcode)
head(final_FGR_alpha_tp1_fig)

In [ ]:
final_FGR_alpha_tp1_fig$Group <- relevel(final_FGR_alpha_tp1_fig$Group, ref = "Control")

In [ ]:
final_FGR_alpha_tp2_fig <-  final_FGR_alpha_tp2 %>% select(kit2.faecal_sample.barcode, Shannon, Richness, Evenness, Group) %>%
                            mutate(Timepoint = "TP2") %>% rename(Sample_ID = kit2.faecal_sample.barcode)
head(final_FGR_alpha_tp2_fig)

In [ ]:
final_FGR_alpha_tp2_fig$Group <- relevel(final_FGR_alpha_tp2_fig$Group, ref = "Control")

### Plots

#### Figure-2

In [ ]:
figure_2 <- ggplot(
  adjust_alpha_output,
  aes(
    x = Alpha_diversity,
    y = OR_inverted,
    ymin = CI_lower_inverted,
    ymax = CI_upper_inverted,
    colour = plot_colour
  )
) +
  geom_pointrange(
    linewidth = 0.6,
    size = 0.8
  ) +
  
  # Reference line
  geom_hline(
    yintercept = 1,
    linetype = "dashed",
    linewidth = 0.5
  ) +
  
  # Separating lines between alpha-diversity measures
  geom_vline(
    xintercept = c(1.5, 2.5),
    colour = "grey80",
    linewidth = 0.6
  ) +
# Panel tags
  geom_text(
    data = tags,
    aes(x = -Inf, y = Inf, label = label),
    inherit.aes = FALSE,
    hjust = -0.6,
    vjust = 1.4,
    fontface = "bold",
    size = 4
  ) +
  
  facet_grid(Risk ~ Timepoint) +  
  facet_grid(Risk ~ Timepoint) +
  
  scale_y_continuous(
    limits = c(y_min, y_max),
    breaks = seq(y_min, y_max, 0.25),
    expand = expansion(mult = c(0.02, 0.05))
  ) +
  
  scale_colour_manual(
    values = c(
      "SGA p <= 0.05" = "#4A9ED9",
      "FGR p <= 0.05" = "#D88999",
      "p > 0.05" = "grey50"
    )
  ) +
  
  labs(
    title = "Adjusted Associations Between Alpha Diversity and Risk of SGA and FGR",
    x = "Alpha diversity",
    y = "Inverted Odds Ratio",
    colour = NULL
  ) +
 
  theme_classic(base_size = 12) +
  theme(
    plot.title = element_text(
      face = "bold",
      size = 14,
      hjust = 0.5
    ),
    panel.border = element_rect(
      colour = "grey50",
      fill = NA,
      linewidth = 0.7
    ),
    panel.spacing = unit(1, "lines"),
    strip.background = element_rect(
      fill = "grey95",
      colour = "grey50"
    ),
    strip.text = element_text(
      face = "bold",
      size = 11
    ) 
  ) 
figure_2

In [ ]:
figure_2 <- ggplot(
  adjust_alpha_output,
  aes(
    x = OR_inverted,
    y = Alpha_diversity,
    xmin = CI_lower_inverted,
    xmax = CI_upper_inverted,
    colour = plot_colour
  )
) +
  geom_pointrange(
    orientation = "y",
    linewidth = 0.6,
    size = 0.8
  ) +
  
  # Reference line - now vertical
  geom_vline(
    xintercept = 1,
    linetype = "dashed",
    linewidth = 0.5
  ) +
  
  # Separating lines between alpha-diversity measures - now horizontal
  geom_hline(
    yintercept = c(1.5, 2.5),
    colour = "grey80",
    linewidth = 0.6
  ) +
  
  geom_text(
    data = tags,
    aes(x = -Inf, y = Inf, label = label),
    inherit.aes = FALSE,
    hjust = 1.6,
    vjust = 1.6,
    fontface = "bold",
    size = 4
  ) +
  
  facet_grid(Risk ~ Timepoint) +
  coord_cartesian(clip = "off") +
  scale_x_continuous(
    limits = c(y_min, y_max),
    breaks = seq(y_min, y_max, 0.25),
    expand = expansion(mult = c(0.02, 0.05))
  ) +
  
  scale_colour_manual(
    values = c(
      "SGA p <= 0.05" = "#4A9ED9",
      "FGR p <= 0.05" = "#D88999",
      "p > 0.05" = "grey50"
    )
  ) +
  
  labs(
    title = "Adjusted Associations Between Alpha Diversity and Risk of SGA and FGR",
    x = "Inverted Odds Ratio",
    y = "Alpha diversity",
    colour = NULL
  ) +
  
  theme_classic(base_size = 12) +
  theme(
      
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    panel.border = element_rect(colour = "grey50", fill = NA, linewidth = 0.7),
    #panel.spacing = unit(1, "lines"),
panel.spacing.x = unit(2, "lines"),
  panel.spacing.y = unit(1, "lines"),
    strip.background = element_rect(fill = "grey95", colour = "grey50"),
    strip.text = element_text(face = "bold", size = 11)
  )

In [ ]:
figure_2

In [ ]:
ggsave("Figure_2.pdf", figure_2,
       width = 10, height = 8, units = "in",
       device = cairo_pdf)



In [ ]:
ggsave("Figure_2.eps", figure_2,
       width = 10, height = 8, units = "in",
       device = cairo_ps)

#### S2

In [ ]:
shared_theme <- theme(
  axis.title.x = element_text(size = 14, color = "black", face = "bold"),
  axis.text.x  = element_text(size = 12, color = "black"),
  axis.title.y = element_text(size = 14, color = "black", face = "bold"),
  axis.text.y  = element_text(size = 12, color = "black"),
  plot.title   = element_text(size = 14, color = "black", face = "bold"),
  legend.title = element_text(size = 14, color = "black", face = "bold"),
  legend.text  = element_text(size = 12, color = "black")
)

In [ ]:
#Shannon diversity Violin plot
plot_shan_tp1_SGA <- ggplot(final_SGA_alpha_tp1_fig, aes(x = Group, y = Shannon, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Shannon Diversity Index", x = "Group", y = "Shannon index") +
  theme_minimal() +
  shared_theme

plot_p_shan_tp1_SGA <- plot_shan_tp1_SGA +
  annotate("text", x = 1.3, y = 4.99,
           label = paste("p =", round(p_value_SGA_shan_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 4.9, yend = 4.9),
               color = "black", size = 0.4)

#plot_p_shan_tp1_SGA


In [ ]:
#Richness diversity Violin plot
plot_rich_tp1_SGA <- ggplot(final_SGA_alpha_tp1_fig, aes(x = Group, y = Richness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Observed Species", x = "Group", y = "Species richness") +
  theme_minimal() +
  shared_theme

plot_p_rich_tp1_SGA <- plot_rich_tp1_SGA +
  annotate("text", x = 1.3, y = 552,
           label = paste("p =", round(p_value_SGA_rich_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 538, yend = 538),
               color = "black", size = 0.4)

#plot_p_rich_tp1_SGA 


In [ ]:
#Evenness Violin plot
plot_even_tp1_SGA <- ggplot(final_SGA_alpha_tp1_fig, aes(x = Group, y = Evenness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Pielou's Evenness", x = "Group", y = "Evenness") +
  theme_minimal() +
  shared_theme

plot_p_even_tp1_SGA <- plot_even_tp1_SGA +
annotate("text", x = 1.30, y = 0.845,
           label = paste("p =", round(p_value_SGA_even_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 0.835, yend = 0.835),
               color = "black", size = 0.4)

#plot_p_even_tp1_SGA 

In [ ]:
#Shannon diversity Violin plot
plot_shan_tp2_SGA <- ggplot(final_SGA_alpha_tp2_fig, aes(x = Group, y = Shannon, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Shannon Diversity Index", x = "Group", y = "Shannon index") +
  theme_minimal() +
  shared_theme

plot_p_shan_tp2_SGA <- plot_shan_tp2_SGA +
  annotate("text", x = 1.3, y = 5.05,
           label = paste("p =", round(p_value_SGA_shan_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 4.95, yend = 4.95),
               color = "black", size = 0.4)

#plot_p_shan_tp2_SGA

In [ ]:
#Richness diversity Violin plot
plot_rich_tp2_SGA <- ggplot(final_SGA_alpha_tp2_fig, aes(x = Group, y = Richness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Observed Species", x = "Group", y = "Species richness") +
  theme_minimal() +
  shared_theme

plot_p_rich_tp2_SGA <- plot_rich_tp2_SGA +
  annotate("text", x = 1.3, y = 552,
           label = paste("p =", round(p_value_SGA_rich_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 538, yend = 538),
               color = "black", size = 0.4)

#plot_p_rich_tp2_SGA 


In [ ]:
#Evenness Violin plot
plot_even_tp2_SGA <- ggplot(final_SGA_alpha_tp2_fig, aes(x = Group, y = Evenness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-SGA" = "#4A9ED9")) +
  labs(title = "Pielou's Evenness", x = "Group", y = "Evenness") +
  theme_minimal() +
  shared_theme

plot_p_even_tp2_SGA <- plot_even_tp2_SGA +
annotate("text", x = 1.30, y = 0.83,
           label = paste("p =", round(p_value_SGA_even_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 0.82, yend = 0.82),
               color = "black", size = 0.4)

#plot_p_even_tp2_SGA 

In [ ]:
#Shannon diversity Violin plot
plot_shan_tp1_FGR <- ggplot(final_FGR_alpha_tp1_fig, aes(x = Group, y = Shannon, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Shannon Diversity Index", x = "Group", y = "Shannon index") +
  theme_minimal() +
  shared_theme

plot_p_shan_tp1_FGR <- plot_shan_tp1_FGR +
  annotate("text", x = 1.3, y = 4.99,
           label = paste("p =", round(p_value_FGR_shan_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 4.9, yend = 4.9),
               color = "black", size = 0.4)

#plot_p_shan_tp1_FGR


In [ ]:
#Richness diversity Violin plot
plot_rich_tp1_FGR <- ggplot(final_FGR_alpha_tp1_fig, aes(x = Group, y = Richness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Observed Species", x = "Group", y = "Species richness") +
  theme_minimal() +
  shared_theme

plot_p_rich_tp1_FGR <- plot_rich_tp1_FGR +
  annotate("text", x = 1.3, y = 552,
           label = paste("p =", round(p_value_FGR_rich_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 538, yend = 538),
               color = "black", size = 0.4)

#plot_p_rich_tp1_FGR 

In [ ]:
#Evenness Violin plot
plot_even_tp1_FGR <- ggplot(final_FGR_alpha_tp1_fig, aes(x = Group, y = Evenness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Pielou's Evenness", x = "Group", y = "Evenness") +
  theme_minimal() +
  shared_theme

plot_p_even_tp1_FGR <- plot_even_tp1_FGR +
annotate("text", x = 1.30, y = 0.845,
           label = paste("p =", round(p_value_FGR_even_tp1, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 0.835, yend = 0.835),
               color = "black", size = 0.4)

#plot_p_even_tp1_FGR

In [ ]:
#Shannon diversity Violin plot
plot_shan_tp2_FGR <- ggplot(final_FGR_alpha_tp2_fig, aes(x = Group, y = Shannon, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Shannon Diversity Index", x = "Group", y = "Shannon index") +
  theme_minimal() +
  shared_theme

plot_p_shan_tp2_FGR <- plot_shan_tp2_FGR +
  annotate("text", x = 1.3, y = 5.35,
           label = paste("p =", round(p_value_FGR_shan_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 5.2, yend = 5.2),
               color = "black", size = 0.4)

#plot_p_shan_tp2_FGR

In [ ]:
#Richness diversity Violin plot
plot_rich_tp2_FGR <- ggplot(final_FGR_alpha_tp2_fig, aes(x = Group, y = Richness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Observed Species", x = "Group", y = "Species richness") +
  theme_minimal() +
  shared_theme

plot_p_rich_tp2_FGR <- plot_rich_tp2_FGR +
  annotate("text", x = 1.3, y = 565,
           label = paste("p =", round(p_value_FGR_rich_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 542, yend = 542),
               color = "black", size = 0.4)

#plot_p_rich_tp2_FGR


In [ ]:
#Evenness Violin plot
plot_even_tp2_FGR <- ggplot(final_FGR_alpha_tp2_fig, aes(x = Group, y = Evenness, fill = Group)) +
  geom_violin(trim = FALSE) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  scale_fill_manual(values = c("Control" = "#708090", "Case-FGR" = "#D88999")) +
  labs(title = "Pielou's Evenness", x = "Group", y = "Evenness") +
  theme_minimal() +
  shared_theme

plot_p_even_tp2_FGR <- plot_even_tp2_FGR +
annotate("text", x = 1.30, y = 0.83,
           label = paste("p =", round(p_value_FGR_even_tp2, 3)),
           color = "black", size = 4.5, hjust = 0, fontface = "bold") +
  geom_segment(aes(x = 1, xend = 2, y = 0.81, yend = 0.81),
               color = "black", size = 0.4)

#plot_p_even_tp2_FGR

In [ ]:

grp_cols <- c("Control"  = "#5B6770",
              "Case-SGA" = "#7FC4E8",
              "Case-FGR" = "#F4B8C1")

legend_df <- data.frame(
  Group = factor(names(grp_cols), levels = names(grp_cols)),
  x = 1, y = 1
)

legend_src <- ggplot(legend_df, aes(x, y, fill = Group)) +
  geom_col() +
  scale_fill_manual(values = grp_cols, name = NULL) +
  theme_classic(base_size = 14) +
  theme(legend.position  = "bottom",
        legend.direction = "horizontal",
        legend.key.size  = unit(0.7, "cm"))

shared_legend <- get_plot_component(legend_src, "guide-box-bottom", return_all = TRUE)[[1]]

In [ ]:
combine_plot <- function(p1, p2, p3, heading) {
  wrap_elements(
    (p1 | p2 | p3) +
      plot_annotation(
        title = heading,
        theme = theme(plot.title = element_text(face = "bold", size = 16, hjust = 0))
      ) &
      theme(legend.position = "none")
  )
}

In [ ]:
panel_A <- combine_plot(plot_p_shan_tp1_SGA, plot_p_rich_tp1_SGA, plot_p_even_tp1_SGA,
                      "A.  SGA vs AGA — TP1")
panel_B <-combine_plot(plot_p_shan_tp2_SGA, plot_p_rich_tp2_SGA, plot_p_even_tp2_SGA,
                      "B.  SGA vs AGA — TP2")
panel_C <- combine_plot(plot_p_shan_tp1_FGR, plot_p_rich_tp1_FGR, plot_p_even_tp1_FGR,
                      "C.  FGR vs AGA — TP1")
panel_D <- combine_plot(plot_p_shan_tp2_FGR, plot_p_rich_tp2_FGR, plot_p_even_tp2_FGR,
                      "D.  FGR vs AGA — TP2")





In [ ]:
legend_only <- legend_src +
  theme(
    axis.title = element_blank(), axis.text = element_blank(),
    axis.ticks = element_blank(), axis.line = element_blank(),
    panel.background = element_blank(),
    plot.background = element_blank()
  ) +
  coord_cartesian(xlim = c(10, 11))  

figure_S2 <- (panel_A | panel_B) /
            (panel_C | panel_D) /
            legend_only +
  plot_layout(heights = c(1, 1, 0.08))

In [ ]:
figure_S2

In [ ]:
ggsave("figure_S2.pdf", figure_S2,
       width = 20, height = 22, units = "in", device = cairo_pdf)

In [ ]:
ggsave("figure_S2.eps", figure_S2,
       width = 20, height = 22, units = "in",
       device = cairo_ps)